<a href="https://colab.research.google.com/github/Rasya-ai-web/Machine-Learning-Lab/blob/main/ML_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Decision Tree Classification and Performance Evaluation Using Confusion Matrix

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

LOAD DATASET

In [5]:
df = pd.read_csv("/content/student_performance_balanced_1000.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

Dataset loaded successfully!
Dataset Shape: (1000, 12)


DATA PREPROCESSING

In [6]:

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent", regex=False)
)

print("\nColumn Names:")
print(df.columns.tolist())

# Remove unnecessary columns
df.drop(
    columns=["studentid", "name"],
    errors="ignore",
    inplace=True
)

# Replace infinity values
df.replace([np.inf, -np.inf], np.nan, inplace=True)


Column Names:
['studentid', 'name', 'attendancerate', 'studyhoursperweek', 'previousgrade', 'assignmentscore', 'finalgrade', 'performance', 'internalmarks', 'attendancepercent', 'studyconsistency', 'participationscore']


HANDLE INVALID VALUES

In [7]:
for column in ["attendancerate", "attendance_percent"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

for column in ["previousgrade", "finalgrade"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

for column in ["studyhoursperweek", "study_hours"]:
    if column in df.columns:
        df.loc[
            df[column] < 0,
            column
        ] = np.nan

REMOVE DUPLICATES

In [8]:
df.drop_duplicates(inplace=True)

print("\nDataset Shape after removing duplicates:", df.shape)


Dataset Shape after removing duplicates: (1000, 10)


CHECK FINAL GRADE

In [9]:

print("\nFinal Grade Statistics:")
print(df["finalgrade"].describe())

print("\nNumber of grades below 50:")
print((df["finalgrade"] < 50).sum())


Final Grade Statistics:
count    1000.000000
mean       63.396190
std        20.536883
min        30.140000
25%        44.870000
50%        63.135000
75%        81.247500
max        99.930000
Name: finalgrade, dtype: float64

Number of grades below 50:
334


 CREATE PERFORMANCE CLASSES

In [10]:

def performance_class(grade):

    if grade < 50:
        return "Fail"

    elif grade < 75:
        return "Average"

    else:
        return "Good"


df["performance"] = df["finalgrade"].apply(
    performance_class
)

print("\n===================================")
print("PERFORMANCE CLASS DISTRIBUTION")
print("===================================")

print(df["performance"].value_counts())


PERFORMANCE CLASS DISTRIBUTION
performance
Fail       334
Average    333
Good       333
Name: count, dtype: int64


 CHECK WHETHER ALL 3 CLASSES EXIST

In [12]:
required_classes = ["Fail", "Average", "Good"]

missing_classes = [
    cls for cls in required_classes
    if cls not in df["performance"].unique()
]

if missing_classes:

    print("\nWARNING!")
    print("The following class/classes are missing:")
    print(missing_classes)

    print("\nYour dataset does not contain enough examples")
    print("for the model to learn these classes.")

    print("\nCurrent classes:")
    print(df["performance"].value_counts())

else:

    print("\nAll three classes are present in the dataset.")


All three classes are present in the dataset.


### Separate Features and Target

In [13]:
X = df.drop(columns=["performance", "finalgrade"])
y = df["performance"]

print("Shape of features (X):", X.shape)
print("Shape of target (y):", y.shape)

Shape of features (X): (1000, 8)
Shape of target (y): (1000,)


ENCODING

In [14]:
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)

TRAIN-TEST SPLIT

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)


Training data: (800, 8)
Testing data : (200, 8)


HANDLE MISSING VALUES

In [16]:

train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

 TRAIN DECISION TREE CLASSIFIER

In [17]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

PREDICT

In [18]:
y_pred = dt_model.predict(X_test)

CONFUSION MATRIX

In [19]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Fail", "Average", "Good"]
)

print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")

print(cm)


CONFUSION MATRIX
[[65  2  0]
 [ 0 66  1]
 [ 0  0 66]]


ACCURACY

In [20]:

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n===================================")
print("ACCURACY")
print("===================================")

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")



ACCURACY
Accuracy: 0.985
Accuracy Percentage: 98.5 %


CLASSIFICATION REPORT

In [21]:
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")

print(
    classification_report(
        y_test,
        y_pred,
        labels=["Fail", "Average", "Good"],
        zero_division=0
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fail       1.00      0.97      0.98        67
     Average       0.97      0.99      0.98        67
        Good       0.99      1.00      0.99        66

    accuracy                           0.98       200
   macro avg       0.99      0.99      0.99       200
weighted avg       0.99      0.98      0.98       200



SAMPLE OUTPUT

In [22]:
print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================")

sample_output = pd.DataFrame({
    "Actual": y_test.iloc[:10].values,
    "Predicted": y_pred[:10]
})

print(sample_output)


SAMPLE PREDICTIONS
    Actual Predicted
0  Average   Average
1     Good      Good
2     Fail      Fail
3     Fail      Fail
4     Good      Good
5     Fail      Fail
6     Good      Good
7  Average   Average
8  Average   Average
9  Average   Average
